# Model Training Pipeline
This pipeline is used for traininig the folowing list of regression models to predict the concentration inmicro mollar, from a given DPV voltammogram signal.

### Lois of regression models:
- Linear Regression
- Ridge Regression
- Elastic Net
- Support Vector Regression
- Decision Tree
- Random Forest
- Extreme Gradient Boost


In [2]:
# imports
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

In [ ]:
df = pd.read_csv('vectorized/core.csv')
df.head()

,sig_id,peak_current,peak_potential,peak_AUC,peak_FWHM,concentration
0,0,34.861237,-0.377680,2.857497,0.082198,100.0
1,1,34.122628,-0.382515,2.858868,0.082198,100.0
2,2,33.952617,-0.382515,2.788808,0.077363,100.0
3,3,22.310970,-0.382515,1.527314,0.062857,50.0
4,4,23.119320,-0.382515,1.600881,0.062857,50.0


In [5]:
X = df.drop(['sig_id', 'concentration'], axis=1)
y = df['concentration']
print(X.shape)
print(y.shape)

(40, 4)
(40,)


In [9]:
loo = LeaveOneOut()
model = LinearRegression()

y_true = []
y_pred = []

for train_idx, test_idx in loo.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    pred = model.predict(X_test)[0]

    y_true.append(y_test.iloc[0])
    y_pred.append(pred)

# Convert to arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("=== Linear Regression with LOOCV ===")
print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

# Per-sample prediction table
results_lr = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "residual": y_true - y_pred,
    "abs_error": np.abs(y_true - y_pred)
})

display(results_lr)

=== Linear Regression with LOOCV ===
MAE  : 0.924852
RMSE : 1.306779
R²   : 0.997727


,y_true,y_pred,residual,abs_error
0,100.00,99.790021,0.209979,0.209979
1,100.00,101.321147,-1.321147,1.321147
2,100.00,97.079781,2.920219,2.920219
3,50.00,51.107950,-1.107950,1.107950
4,50.00,54.444805,-4.444805,4.444805
5,50.00,52.037904,-2.037904,2.037904
6,25.00,23.248117,1.751883,1.751883
7,25.00,23.021035,1.978965,1.978965
8,25.00,23.072405,1.927595,1.927595
9,15.00,13.748714,1.251286,1.251286


In [10]:
outer_cv = LeaveOneOut()

y_true = []
y_pred = []
best_params_per_fold = []

for train_idx, test_idx in outer_cv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Inner CV for hyperparameter tuning
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

    model = DecisionTreeRegressor(random_state=42)

    param_grid = {
        "max_depth": [2, 3, 4, 5, None],
        "min_samples_split": [2, 3, 4, 5, 6],
        "min_samples_leaf": [1, 2, 3, 4],
        "ccp_alpha": [0.0, 0.001, 0.01, 0.05]
    }

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_

    pred = best_model.predict(X_test)[0]

    y_true.append(y_test.iloc[0])
    y_pred.append(pred)
    best_params_per_fold.append(grid.best_params_)

# Convert to arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("=== DecisionTreeRegressor with nested LOOCV ===")
print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

# Per-sample prediction table
results_dtr = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "residual": y_true - y_pred,
    "abs_error": np.abs(y_true - y_pred)
})

display(results_dtr)

# Best hyperparameters chosen in each outer fold
best_params_dtr = pd.DataFrame(best_params_per_fold)
display(best_params_dtr)

# Optional: most frequent best parameter combination
print("\nMost frequent best parameter sets:")
display(best_params_dtr.value_counts().reset_index(name="count"))

=== DecisionTreeRegressor with nested LOOCV ===
MAE  : 0.062500
RMSE : 0.395285
R²   : 0.999792


,y_true,y_pred,residual,abs_error
0,100.00,100.00,0.0,0.0
1,100.00,100.00,0.0,0.0
2,100.00,100.00,0.0,0.0
3,50.00,50.00,0.0,0.0
4,50.00,50.00,0.0,0.0
5,50.00,50.00,0.0,0.0
6,25.00,25.00,0.0,0.0
7,25.00,25.00,0.0,0.0
8,25.00,25.00,0.0,0.0
9,15.00,15.00,0.0,0.0


,ccp_alpha,max_depth,min_samples_leaf,min_samples_split
0,0.0,NaN,1,4
1,0.0,NaN,1,4
2,0.0,NaN,1,4
3,0.0,NaN,1,2
4,0.0,NaN,1,2
5,0.0,NaN,1,2
6,0.0,NaN,1,2
7,0.0,NaN,1,2
8,0.0,NaN,1,2
9,0.0,NaN,1,2



Most frequent best parameter sets:


,ccp_alpha,max_depth,min_samples_leaf,min_samples_split,count
0,0.0,4.0,1,2,2
1,0.0,5.0,1,2,1


In [11]:
outer_cv = LeaveOneOut()

y_true = []
y_pred = []
best_params_per_fold = []

for train_idx, test_idx in outer_cv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Inner CV for hyperparameter tuning
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

    model = Ridge(random_state=42)

    param_grid = {
        "alpha": [0.001, 0.01, 0.1, 1, 10, 100],
        "fit_intercept": [True, False]
    }

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_

    pred = best_model.predict(X_test)[0]

    y_true.append(y_test.iloc[0])
    y_pred.append(pred)
    best_params_per_fold.append(grid.best_params_)

# Convert to arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("=== Ridge with nested LOOCV ===")
print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

# Per-sample prediction table
results_dtr = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "residual": y_true - y_pred,
    "abs_error": np.abs(y_true - y_pred)
})

display(results_dtr)

# Best hyperparameters chosen in each outer fold
best_params_dtr = pd.DataFrame(best_params_per_fold)
display(best_params_dtr)

# Optional: most frequent best parameter combination
print("\nMost frequent best parameter sets:")
display(best_params_dtr.value_counts().reset_index(name="count"))

=== Ridge with nested LOOCV ===
MAE  : 1.004226
RMSE : 1.698293
R²   : 0.996162


,y_true,y_pred,residual,abs_error
0,100.00,97.660557,2.339443,2.339443
1,100.00,97.461627,2.538373,2.538373
2,100.00,94.420973,5.579027,5.579027
3,50.00,53.829268,-3.829268,3.829268
4,50.00,54.532687,-4.532687,4.532687
5,50.00,55.035032,-5.035032,5.035032
6,25.00,24.932212,0.067788,0.067788
7,25.00,24.786270,0.213730,0.213730
8,25.00,24.866714,0.133286,0.133286
9,15.00,14.002918,0.997082,0.997082


,alpha,fit_intercept
0,0.100,False
1,0.100,False
2,0.100,False
3,0.100,False
4,0.001,True
5,0.100,False
6,0.100,False
7,0.100,False
8,0.100,False
9,0.100,False



Most frequent best parameter sets:


,alpha,fit_intercept,count
0,0.100,False,39
1,0.001,True,1


In [12]:
outer_cv = LeaveOneOut()

y_true = []
y_pred = []
best_params_per_fold = []

for train_idx, test_idx in outer_cv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Inner CV for hyperparameter tuning
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

    model = ElasticNet(random_state=42, max_iter=20000)

    param_grid = {
        "alpha": [1e-4, 1e-3, 1e-2, 1e-1, 1, 10],
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
        "fit_intercept": [True, False]
    }

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_

    pred = best_model.predict(X_test)[0]

    y_true.append(y_test.iloc[0])
    y_pred.append(pred)
    best_params_per_fold.append(grid.best_params_)

# Convert to arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("=== Ridge with nested LOOCV ===")
print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

# Per-sample prediction table
results_dtr = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "residual": y_true - y_pred,
    "abs_error": np.abs(y_true - y_pred)
})

display(results_dtr)

# Best hyperparameters chosen in each outer fold
best_params_dtr = pd.DataFrame(best_params_per_fold)
display(best_params_dtr)

# Optional: most frequent best parameter combination
print("\nMost frequent best parameter sets:")
display(best_params_dtr.value_counts().reset_index(name="count"))

=== Ridge with nested LOOCV ===
MAE  : 0.949602
RMSE : 1.645826
R²   : 0.996395


,y_true,y_pred,residual,abs_error
0,100.00,97.211014,2.788986,2.788986
1,100.00,96.805793,3.194207,3.194207
2,100.00,93.986330,6.013670,6.013670
3,50.00,52.852102,-2.852102,2.852102
4,50.00,54.497418,-4.497418,4.497418
5,50.00,53.907864,-3.907864,3.907864
6,25.00,25.232553,-0.232553,0.232553
7,25.00,25.097926,-0.097926,0.097926
8,25.00,25.182973,-0.182973,0.182973
9,15.00,14.106086,0.893914,0.893914


,alpha,fit_intercept,l1_ratio
0,0.0100,False,0.7
1,0.0100,False,0.7
2,0.0100,False,0.7
3,0.0100,False,0.9
4,0.0001,True,0.9
5,0.0100,False,0.9
6,0.0100,False,0.7
7,0.0100,False,0.7
8,0.0100,False,0.7
9,0.0100,False,0.7



Most frequent best parameter sets:


,alpha,fit_intercept,l1_ratio,count
0,0.0100,False,0.7,37
1,0.0100,False,0.9,2
2,0.0001,True,0.9,1


In [13]:
outer_cv = LeaveOneOut()

y_true = []
y_pred = []
best_params_per_fold = []

for train_idx, test_idx in outer_cv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Inner CV for hyperparameter tuning
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

    model = SVR()

    param_grid = {
        "kernel": ["rbf", "linear"],
        "C": [0.1, 1, 10, 100],
        "epsilon": [0.01, 0.05, 0.1, 0.5, 1.0],
        "gamma": ["scale", "auto"]
    }

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_

    pred = best_model.predict(X_test)[0]

    y_true.append(y_test.iloc[0])
    y_pred.append(pred)
    best_params_per_fold.append(grid.best_params_)

# Convert to arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("=== Ridge with nested LOOCV ===")
print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

# Per-sample prediction table
results_dtr = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "residual": y_true - y_pred,
    "abs_error": np.abs(y_true - y_pred)
})

display(results_dtr)

# Best hyperparameters chosen in each outer fold
best_params_dtr = pd.DataFrame(best_params_per_fold)
display(best_params_dtr)

# Optional: most frequent best parameter combination
print("\nMost frequent best parameter sets:")
display(best_params_dtr.value_counts().reset_index(name="count"))

=== Ridge with nested LOOCV ===
MAE  : 2.342918
RMSE : 6.415718
R²   : 0.945221


,y_true,y_pred,residual,abs_error
0,100.00,78.354696,21.645304,21.645304
1,100.00,76.719992,23.280008,23.280008
2,100.00,76.330393,23.669607,23.669607
3,50.00,52.579888,-2.579888,2.579888
4,50.00,56.298720,-6.298720,6.298720
5,50.00,54.615780,-4.615780,4.615780
6,25.00,25.117623,-0.117623,0.117623
7,25.00,24.955873,0.044127,0.044127
8,25.00,25.028432,-0.028432,0.028432
9,15.00,14.135255,0.864745,0.864745


,C,epsilon,gamma,kernel
0,0.1,0.50,scale,linear
1,0.1,0.50,scale,linear
2,0.1,0.50,scale,linear
3,100.0,1.00,scale,linear
4,100.0,0.01,scale,linear
5,100.0,0.01,scale,linear
6,100.0,0.05,scale,linear
7,100.0,0.05,scale,linear
8,100.0,0.05,scale,linear
9,100.0,0.05,scale,linear



Most frequent best parameter sets:


,C,epsilon,gamma,kernel,count
0,100.0,0.01,scale,linear,27
1,100.0,0.05,scale,linear,5
2,100.0,0.10,scale,linear,4
3,0.1,0.50,scale,linear,3
4,100.0,1.00,scale,linear,1


In [15]:
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

outer_cv = LeaveOneOut()

y_true = []
y_pred = []
best_params_per_fold = []

for train_idx, test_idx in outer_cv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Inner CV for hyperparameter tuning
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

    model = RandomForestRegressor(random_state=42, n_jobs=-1)

    search_spaces = {
        "n_estimators": Integer(100, 500),
        "max_depth": Integer(2, 10),
        "min_samples_split": Integer(2, 8),
        "min_samples_leaf": Integer(1, 4),
        "max_features": Categorical([1.0, "sqrt", 0.75])
    }

    opt = BayesSearchCV(
        estimator=model,
        search_spaces=search_spaces,
        n_iter=25,
        cv=inner_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        random_state=42,
        refit=True
    )

    opt.fit(X_train, y_train)
    best_model = opt.best_estimator_

    pred = best_model.predict(X_test)[0]

    y_true.append(y_test.iloc[0])
    y_pred.append(pred)
    best_params_per_fold.append(opt.best_params_)

# Convert to arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("=== RandomForestRegressor with nested LOOCV + BayesSearchCV ===")
print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

# Per-sample prediction table
results_rfr = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "residual": y_true - y_pred,
    "abs_error": np.abs(y_true - y_pred)
})

display(results_rfr)

# Best hyperparameters chosen in each outer fold
best_params_rfr = pd.DataFrame(best_params_per_fold)
display(best_params_rfr)

# Optional: most frequent best parameter combination
print("\nMost frequent best parameter sets:")
display(best_params_rfr.value_counts().reset_index(name="count"))

/Users/andrei/Work/PhD/pyocyanin-ai/.venv/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(2), np.float64(1.0), np.int64(1), np.int64(2), np.int64(500)] before, using random point [np.int64(2), 0.75, np.int64(3), np.int64(2), np.int64(386)]
  warnings.warn(
/Users/andrei/Work/PhD/pyocyanin-ai/.venv/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(2), np.float64(1.0), np.int64(1), np.int64(2), np.int64(500)] before, using random point [np.int64(8), 1.0, np.int64(1), np.int64(6), np.int64(458)]
  warnings.warn(
/Users/andrei/Work/PhD/pyocyanin-ai/.venv/lib/python3.12/site-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [np.int64(2), np.float64(1.0), np.int64(1), np.int64(2), np.int64(500)] before, using random point [np.int64(7), 1.0, np.int64(3), np.int64(5), np.int64(316)]
  warnin

=== RandomForestRegressor with nested LOOCV + BayesSearchCV ===
MAE  : 0.982894
RMSE : 2.213000
R²   : 0.993482


,y_true,y_pred,residual,abs_error
0,100.00,94.969512,5.030488,5.030488
1,100.00,91.311475,8.688525,8.688525
2,100.00,91.337386,8.662614,8.662614
3,50.00,47.866324,2.133676,2.133676
4,50.00,52.460000,-2.460000,2.460000
5,50.00,49.290000,0.710000,0.710000
6,25.00,23.824885,1.175115,1.175115
7,25.00,23.824885,1.175115,1.175115
8,25.00,23.824885,1.175115,1.175115
9,15.00,14.319809,0.680191,0.680191


,max_depth,max_features,min_samples_leaf,min_samples_split,n_estimators
0,10,0.75,1,2,328
1,10,0.75,1,2,305
2,10,0.75,1,2,329
3,10,0.75,1,2,389
4,7,0.75,1,2,500
5,10,0.75,1,2,500
6,10,0.75,1,2,434
7,10,0.75,1,2,434
8,10,0.75,1,2,434
9,10,0.75,1,2,419



Most frequent best parameter sets:


,max_depth,max_features,min_samples_leaf,min_samples_split,n_estimators,count
0,9,0.75,1,2,500,5
1,10,0.75,1,2,434,3
2,9,1.00,1,2,500,3
3,8,1.00,1,2,500,3
4,10,0.75,1,2,500,2
5,10,0.75,1,2,392,2
6,10,0.75,1,2,418,2
7,10,0.75,1,2,395,2
8,5,1.00,1,2,500,2
9,10,0.75,1,2,328,1


In [16]:
from skopt import BayesSearchCV
from skopt.space import Integer, Real

outer_cv = LeaveOneOut()

y_true = []
y_pred = []
best_params_per_fold = []

for train_idx, test_idx in outer_cv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Inner CV for hyperparameter tuning
    inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=1,
        verbosity=0
    )

    search_spaces = {
        "n_estimators": Integer(50, 500),
        "max_depth": Integer(2, 8),
        "learning_rate": Real(0.01, 0.3, prior="log-uniform"),
        "subsample": Real(0.6, 1.0),
        "colsample_bytree": Real(0.6, 1.0),
        "min_child_weight": Integer(1, 10),
        "gamma": Real(1e-8, 5.0, prior="log-uniform"),
        "reg_alpha": Real(1e-8, 10.0, prior="log-uniform"),
        "reg_lambda": Real(1e-3, 10.0, prior="log-uniform")
    }

    opt = BayesSearchCV(
        estimator=model,
        search_spaces=search_spaces,
        n_iter=25,
        cv=inner_cv,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
        random_state=42,
        refit=True
    )

    opt.fit(X_train, y_train)
    best_model = opt.best_estimator_

    pred = best_model.predict(X_test)[0]

    y_true.append(y_test.iloc[0])
    y_pred.append(pred)
    best_params_per_fold.append(opt.best_params_)

# Convert to arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Metrics
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("=== XGBRegressor with nested LOOCV + BayesSearchCV ===")
print(f"MAE  : {mae:.6f}")
print(f"RMSE : {rmse:.6f}")
print(f"R²   : {r2:.6f}")

# Per-sample prediction table
results_xgb = pd.DataFrame({
    "y_true": y_true,
    "y_pred": y_pred,
    "residual": y_true - y_pred,
    "abs_error": np.abs(y_true - y_pred)
})

display(results_xgb)

# Best hyperparameters chosen in each outer fold
best_params_xgb = pd.DataFrame(best_params_per_fold)
display(best_params_xgb)

# Optional: most frequent best parameter combination
print("\nMost frequent best parameter sets:")
display(best_params_xgb.value_counts().reset_index(name="count"))

=== XGBRegressor with nested LOOCV + BayesSearchCV ===
MAE  : 2.599263
RMSE : 6.507462
R²   : 0.943644


,y_true,y_pred,residual,abs_error
0,100.00,99.999191,0.000809,0.000809
1,100.00,79.253517,20.746483,20.746483
2,100.00,79.253517,20.746483,20.746483
3,50.00,24.285742,25.714258,25.714258
4,50.00,58.069530,-8.069530,8.069530
5,50.00,44.644360,5.355640,5.355640
6,25.00,19.371653,5.628347,5.628347
7,25.00,24.308384,0.691616,0.691616
8,25.00,24.899021,0.100979,0.100979
9,15.00,15.169655,-0.169655,0.169655


,colsample_bytree,gamma,learning_rate,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,subsample
0,0.600000,1.954576e-07,0.300000,4,1,500,6.186570e-07,1.910806,1.000000
1,0.939518,3.391347e-01,0.069808,3,3,235,1.678135e-07,0.100103,0.768286
2,0.939518,3.391347e-01,0.069808,3,3,235,1.678135e-07,0.100103,0.768286
3,0.600000,1.000000e-08,0.300000,6,2,53,1.000000e-08,0.168799,0.901875
4,0.600000,5.000000e+00,0.300000,2,1,443,1.000000e-08,0.028119,0.600000
5,0.600000,1.000000e-08,0.010000,8,1,500,1.328606e-08,0.001000,0.643701
6,0.691977,7.497092e-01,0.063061,5,1,317,1.539071e-07,0.213743,0.745019
7,1.000000,2.289701e+00,0.299971,2,2,500,1.138614e-05,0.001000,0.782984
8,1.000000,2.289701e+00,0.299971,2,2,500,1.138614e-05,0.001000,0.782984
9,0.834024,7.989800e-04,0.038268,8,2,500,1.583671e-08,0.035269,0.936406



Most frequent best parameter sets:


,colsample_bytree,gamma,learning_rate,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,subsample,count
0,0.939518,3.391347e-01,0.069808,3,3,235,1.678135e-07,0.100103,0.768286,2
1,1.000000,2.289701e+00,0.299971,2,2,500,1.138614e-05,0.001000,0.782984,2
2,1.000000,3.022664e-01,0.025408,4,1,328,1.000000e-08,0.001000,0.836936,2
3,1.000000,1.000000e-08,0.300000,2,1,265,1.000000e-08,10.000000,0.600000,2
4,0.600000,1.954576e-07,0.300000,4,1,500,6.186570e-07,1.910806,1.000000,1
5,0.600000,1.000000e-08,0.300000,6,2,53,1.000000e-08,0.168799,0.901875,1
6,0.600000,5.000000e+00,0.300000,2,1,443,1.000000e-08,0.028119,0.600000,1
7,0.600000,1.000000e-08,0.010000,8,1,500,1.328606e-08,0.001000,0.643701,1
8,0.691977,7.497092e-01,0.063061,5,1,317,1.539071e-07,0.213743,0.745019,1
9,0.834024,7.989800e-04,0.038268,8,2,500,1.583671e-08,0.035269,0.936406,1
